# 03 · DCF valuation of the acquired business (RemainCo)

Two **separate** information sets. They are never mixed:

| Perspective | Valuation date | Market data | Cash-flow basis |
|---|---|---|---|
| **A** announcement | 14 Dec 2017 | 13 Dec 2017 close | **Estimates** built from public Dec 2017 information (21CF 10-Ks/10-Q; New Fox ~$10bn revenue / ~$2.8bn EBITDA from the 14 Dec release) |
| **B** proxy | 30 May 2018 | 30 May 2018 close | **Management forecasts** in the May 2018 joint proxy (Disney's forecasts for RemainCo as the base case, 21CF's as an alternative) |

The June 2018 updated forecasts and anything later (RSN sale, Sky sale, closing) are *subsequent information*.
They appear only in labelled comparison and hindsight rows. Code: `src/valuation/`. Method: `docs/valuation_methodology.md`.
Every input in `dcf_assumptions.csv` carries a `value_type` (reported / reported_forecast / market / derived / estimate),
a source, an observation date and an availability date. A check fails if any input became available after its perspective's cutoff.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_rows", 300)
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
PROC, FIG = ROOT / "data" / "processed", ROOT / "docs" / "figures"

# Rebuild Stage 3 from stored files (offline). Stage 1/2 outputs are inputs and are not modified.
for script in ("valuation/build_valuation.py", "valuation/charts.py"):
    res = subprocess.run([sys.executable, str(ROOT / "src" / script)], cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
    print(res.stdout.strip()); assert res.returncode == 0, res.stdout + res.stderr

A = pd.read_csv(PROC / "dcf_assumptions.csv")
O = pd.read_csv(PROC / "valuation_outputs.csv"); O["v"] = pd.to_numeric(O.value, errors="coerce")
S = pd.read_csv(PROC / "valuation_sensitivity.csv")
P = pd.read_csv(PROC / "dcf_projections.csv")
V = pd.read_csv(PROC / "verification_checks.csv")

## 1. Management forecasts (extracted with page and table references; footnote markers handled explicitly)

In [ ]:
F = pd.read_csv(PROC / "management_forecasts.csv")
F.pivot_table(index=["proxy", "forecast_set", "metric"], columns="fiscal_year", values="value", aggfunc="first")

In [ ]:
F[["forecast_set", "metric", "source_page", "source_table_index", "definition", "footnote"]].drop_duplicates(["forecast_set", "metric"])

## 2. Assumptions by perspective

In [ ]:
A[~A.category.isin(["forecast"])][["perspective", "category", "parameter", "value", "unit", "value_type", "available_date",
                                   "sensitivity_low", "sensitivity_high", "source", "method"]]

## 3. WACC

In [ ]:
O[O.method == "WACC"].pivot_table(index="metric", columns="perspective", values="v").round(4)

In [ ]:
A[A.category == "wacc_beta"][["perspective", "parameter", "value", "method"]]

## 4. FCFF projections
For B, FCFF equals management's unlevered FCF exactly (checked). EBIT, D&A, taxes, capex and the working-capital residual are a labelled decomposition.

In [ ]:
cols = ["perspective", "case", "fy", "revenue", "ebitda", "ebitda_margin", "da", "ebit", "taxes_on_ebit", "nopat", "capex", "dnwc", "fcff",
        "fraction_after_valuation_date", "discount_time_years", "discount_factor", "pv_fcff"]
P[cols].round(3)

## 5. Enterprise value → equity value

In [ ]:
keep = ["enterprise_value_ops", "pv_explicit_fcff", "pv_terminal_value", "terminal_value_share_of_ev", "plus_sky_39pct_stake",
        "plus_other_investments_book", "less_debt", "plus_cash", "less_noncontrolling_interests", "equity_value", "equity_value_per_21cf_share"]
q = O[O.method.isin(["DCF", "Bridge"]) & O.metric.isin(keep)]
q.pivot_table(index=["perspective", "case"], columns="metric", values="v", aggfunc="first")[[k for k in keep if k in set(q.metric)]].round(2).T

## 6. Sensitivities and scenarios

In [ ]:
for p in S.perspective.unique():
    g = S[(S.perspective == p) & (S.table == "wacc_x_terminal_growth")]
    print(p, "- equity value per 21CF share (USD)")
    display(g.pivot_table(index="row_value", columns="col_value", values="equity_per_share").rename_axis(index="WACC", columns="g"))

In [ ]:
S[S.table != "wacc_x_terminal_growth"][["perspective", "table", "row_value", "enterprise_value_ops", "equity_value", "equity_per_share"]]

In [ ]:
O[(O.method == "DCF scenario") & O.metric.isin(["enterprise_value_ops", "equity_value", "equity_value_per_21cf_share"])].pivot_table(
    index=["perspective", "case"], columns="metric", values="v").round(2)

In [ ]:
display(Image(FIG / "08_valuation_football_field.png"))

## 7. Subsequent information (never used in A or B)

In [ ]:
O[O.perspective.isin(["hindsight", "subsequent_info_2018-06"])][["perspective", "method", "case", "metric", "v", "notes"]].head(40)

## 8. Validation checks

In [ ]:
v = V[V.check_type.str.startswith(("valuation_", "forecast_", "transaction_premium"))]
print(v.groupby(["check_type", "status"]).size())
assert (v.status == "pass").all()
print("Stage 1/2 checks:", V[~V.index.isin(v.index)].groupby("status").size().to_dict())